## Компьютерное зрение в робототехнике | Домашнее задание 4

В этом домашнем задании нужно будет реализовать подсчет числа пальцев, которые показывает человек на видео *fingers.mov*.

Для удобства реализации значительная часть кода уже написана. Внимательно ознакомьтесь с заготовками, поймите, что происходит в коде и для чего нужна каждая конкретная функция.

Есть много вариантов, как можно подступиться к этой задаче. Приведем некоторые из них:

**Скелетонизация**
- получение маски
- обработка (фильтрация шумов, сглаживание)
- удаление всего, кроме самой большой связной компоненты
- скелетонизация (можно загуглить opencv skeletonization)
- нахождение кончиков пальцев (filter2d)
- фильтрация кончиков пальцев

**Дефекты выпуклости**
- получение маски
- обработка (фильтрация шумов, сглаживание)
- удаление всего, кроме самой большой связной компоненты
- нахождение контура, построение грубой аппроксимации
- нахождение дефектов выпуклости (convexity defects)

**Морфология**
- получение маски
- обработка (фильтрация шумов, сглаживание)
- удаление всего, кроме самой большой связной компоненты
- морфологические операции top hat/black hat

Реализуйте все три способа и сравните их. Можно придумать свой и тоже сравнить :)
Если хотите использовать нейронные сети и другие методы машинного обучения, сначала придумайте один метод на классическом зрении.

***

<h2 style="color:#A7BD3F;">Секция 1: подготовка</h2>

Этот код представляет из себя обертку над функцией обработки кадра. Завершение выполнения производится с помощью нажатия кнопки 'q' (работает только для английского языка).

In [1]:
import numpy as np
import cv2

class FrameProcessor:
    def __init__(self):
        pass
    
    def processing_loop(self, source, lth, hth, max_frame_num = -1,\
                        alternative_source="", save_to_file=""):
        i = 0
        results = []

        output_file = None
        
        #out = cv2.VideoWriter('outpy.avi', cv2.VideoWriter_fourcc('M','J','P','G'),
#                               30, (WINDX, WINDY))
#         out.write(canvas)
#         out.release()
        
        while (True):
            retval, frame = source.read()

            if (retval == False):
                print("Cannot read frame")
                
                if (alternative_source != ""):
                    print("Opening alternative source ", alternative_source)
                    source = cv2.VideoCapture(alternative_source)
                    continue
                
                else:
                    print("Exiting loop")
                    break

            result = self.process_frame(frame, lth, hth)
            
            results.append(result)

            key = cv2.waitKey(100) & 0xFF

            i += 1

            if (key == ord('q')):
                break
                        
            if (max_frame_num != -1 and i >= max_frame_num):
                break

        return results
    
    def process_frame(self, frame, lth, hth):
        return 5

***

<h2 style="color:#A7BD3F;">Секция 2: настройка цветового фильтра</h2>

Настройте параметры фильтрации по цвету. Это нужно делать в цветовом пространстве *HSV*. После этого запишите найденные параметры в *lth* и *hth*, их можно будет использовать позже.

In [2]:
#############################################
# YOUR DEFAULT PARAMETERS BELOW
#############################################

lth, hth = (0, 43, 148), (32, 184, 255)

#############################################
# YOUR DEFAULT PARAMETERS ABOVE
#############################################

class ColorFilterTuning(FrameProcessor):
    def __init__(self):
        super().__init__()

        cv2.namedWindow("frame")

        cv2.createTrackbar("H min", "frame", lth[0], 179, self.nothing)
        cv2.createTrackbar("S min", "frame", lth[1], 255, self.nothing)
        cv2.createTrackbar("V min", "frame", lth[2], 255, self.nothing)

        cv2.createTrackbar("H max", "frame", hth[0], 179, self.nothing)
        cv2.createTrackbar("S max", "frame", hth[1], 255, self.nothing)
        cv2.createTrackbar("V max", "frame", hth[2], 255, self.nothing)

    def nothing(self, inp):
        pass

    def process_frame(self, frame, lth, hth):
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        low_th = (
            cv2.getTrackbarPos("H min", "frame"),
            cv2.getTrackbarPos("S min", "frame"),
            cv2.getTrackbarPos("V min", "frame"),
        )

        high_th = (
            cv2.getTrackbarPos("H max", "frame"),
            cv2.getTrackbarPos("S max", "frame"),
            cv2.getTrackbarPos("V max", "frame"),
        )

        mask = cv2.inRange(hsv, low_th, high_th)

        cv2.imshow("frame", frame)
        cv2.imshow("mask", mask)

        return low_th, high_th

In [3]:
import numpy as np
import cv2

video_file = "fingers.mov"

cam = cv2.VideoCapture(video_file)

#print(cam)
# frame_offset = 100
# cam.set(1, frame_offset)

tuner = ColorFilterTuning()

colors = tuner.processing_loop(cam, None, None, max_frame_num = -1,\
            alternative_source=video_file)
lth, hth = colors[-1]

print("Color filter parameters: ", lth, hth)
cam.release()
cv2.destroyAllWindows()
cv2.waitKey(100)

QFontDatabase: Cannot find font directory /home/vlad/Documents/Компьютерное зрение/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/vlad/Documents/Компьютерное зрение/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/vlad/Documents/Компьютерное зрение/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/vlad/Documents/Компьютерное зрение/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) o

Color filter parameters:  (0, 43, 148) (32, 184, 255)


-1

***

<h2 style="color:#A7BD3F;">Секция 3: подсчет пальцев</h2>

Реализуйте алгоритм нахождения числа пальцев, закомментировав функцию, которая всегда возвращает 3 и написав свою. Вы можете использовать *lth* и *hth*, которые получили выше. Заготовленные функции для обработки масок можно использовать, а можно и не использовать.

In [ ]:
def fill_holes(img):
    h, w = img.shape

    img_enlarged = np.zeros((h + 2, w + 2), np.uint8)
    img_enlarged[1 : h + 1, 1 : w + 1] = img

    img_enl_not = cv2.bitwise_not(img_enlarged)

    _, im_th = cv2.threshold(img_enl_not, 220, 255, cv2.THRESH_BINARY_INV)

    im_floodfill = im_th.copy()

    h, w = im_th.shape[:2]
    mask = np.zeros((h + 2, w + 2), np.uint8)

    cv2.floodFill(im_floodfill, mask, (0, 0), 255)

    im_floodfill_inv = cv2.bitwise_not(im_floodfill)
    im_out = im_th | im_floodfill_inv

    result = im_out[1 : h - 1, 1 : w - 1]

    return result


class SkeletonizationCounter(FrameProcessor):
    def __init__(self):
        super().__init__()

    def process_frame(self, frame, lth, hth):
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        mask = cv2.inRange(hsv, lth, hth)

        kernel = np.ones((5, 5), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask)

        hand_mask = np.zeros_like(mask)

        if num_labels > 1:
            largest_label = np.argmax(stats[1:, cv2.CC_STAT_AREA]) + 1
            hand_mask[labels == largest_label] = 255

        hand_mask = fill_holes(hand_mask)
        skeleton = self.get_skeleton(hand_mask)
        endpoints = self.get_endpoints(skeleton)

        # Фильтрацияпо расположению
        endpoints = self.filter_endpoints(skeleton, endpoints)

        # Фильтрация по толщине
        filtered_endpoints = self.filter_thick_endpoints(hand_mask, skeleton, endpoints)
        fingers_num = min(5, cv2.countNonZero(filtered_endpoints))
        result = cv2.cvtColor(skeleton, cv2.COLOR_GRAY2BGR)

        # Красные точки -- удалённые окончания
        removed = cv2.subtract(endpoints, filtered_endpoints)

        ys, xs = np.where(removed > 0)
        for x, y in zip(xs, ys):
            cv2.circle(result, (int(x), int(y)), 4, (0, 0, 255), -1)

        # Зелёные точки -- засчитанные окончания
        ys, xs = np.where(filtered_endpoints > 0)
        for x, y in zip(xs, ys):
            cv2.circle(result, (int(x), int(y)), 4, (0, 255, 0), -1)

        cv2.imshow("hand mask", hand_mask)
        cv2.imshow("skeleton endpoints", result)

        return int(fingers_num)

    def get_skeleton(self, mask):
        mask = cv2.resize(mask, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_NEAREST)

        skeleton = (mask > 0).astype(np.uint8)

        changed = True

        while changed:
            changed = False

            for step in range(2):
                p = np.pad(skeleton, 1)

                neighbors = [
                    p[:-2, 1:-1],
                    p[:-2, 2:],
                    p[1:-1, 2:],
                    p[2:, 2:],
                    p[2:, 1:-1],
                    p[2:, :-2],
                    p[1:-1, :-2],
                    p[:-2, :-2],
                ]

                count = sum(neighbors)

                transitions = sum(
                    (neighbors[i] == 0) & (neighbors[(i + 1) % 8] == 1)
                    for i in range(8)
                )

                remove = (
                    (skeleton == 1) & (count >= 2) & (count <= 6) & (transitions == 1)
                )

                if step == 0:
                    remove &= (neighbors[0] * neighbors[2] * neighbors[4] == 0) & (
                        neighbors[2] * neighbors[4] * neighbors[6] == 0
                    )
                else:
                    remove &= (neighbors[0] * neighbors[2] * neighbors[6] == 0) & (
                        neighbors[0] * neighbors[4] * neighbors[6] == 0
                    )

                if np.any(remove):
                    skeleton[remove] = 0
                    changed = True

        return skeleton * 255

    def get_endpoints(self, skeleton):
        binary = (skeleton > 0).astype(np.uint8)

        kernel = np.ones((3, 3), np.uint8)

        neighbors = cv2.filter2D(
            binary, cv2.CV_16S, kernel, borderType=cv2.BORDER_CONSTANT
        )

        endpoints = ((binary == 1) & (neighbors == 2)).astype(np.uint8) * 255

        return endpoints

    def filter_endpoints(self, skeleton, endpoints):
        ys, xs = np.where(skeleton > 0)

        result = np.zeros_like(endpoints)

        if len(xs) == 0:
            return result

        min_x = np.min(xs)
        max_x = np.max(xs)

        # Рассматриваем правые 30% скелета
        limit_x = min_x + 0.7 * (max_x - min_x)

        ys, xs = np.where(endpoints > 0)

        for x, y in zip(xs, ys):
            if x >= limit_x:
                result[y, x] = 255

        return result

    def filter_thick_endpoints(self, hand_mask, skeleton, endpoints):
        h, w = skeleton.shape

        # Приводим маску руки к размеру скелета
        mask = cv2.resize(hand_mask, (w, h))

        # Расстояние от белых пикселей до фона
        distances = cv2.distanceTransform(mask, cv2.DIST_L2, 5)

        ys, xs = np.where(mask > 0)

        if len(xs) == 0:
            return endpoints.copy()

        # Оцениваем толщину кисти в правой половине руки
        limit_x = int(np.min(xs) + 0.5 * (np.max(xs) - np.min(xs)))

        max_radius = np.max(distances[:, limit_x:])

        result = endpoints.copy()

        ys, xs = np.where(endpoints > 0)

        for x, y in zip(xs, ys):
            radius = distances[y, x]

            # Слишком широкая область -- скорее всего, не палец
            if radius > 0.55 * max_radius:
                result[y, x] = 0

        return result


class ConvexityCounter(FrameProcessor):
    def __init__(self):
        super().__init__()

    def get_defects(self, frame, hand_mask):
        result = frame.copy()

        contours, _ = cv2.findContours(
            hand_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        if len(contours) == 0:
            return result, 0

        contour = max(contours, key=cv2.contourArea)

        if len(contour) < 4:
            return result, 0

        cv2.drawContours(result, [contour], -1, (255, 0, 0), 2)

        hull = cv2.convexHull(contour)

        cv2.drawContours(result, [hull], -1, (0, 255, 0), 2)

        hull_indices = cv2.convexHull(contour, returnPoints=False)

        if hull_indices is None or len(hull_indices) < 3:
            return result, 0

        defects = cv2.convexityDefects(contour, hull_indices)

        if defects is None:
            return result, 0

        xs = contour[:, 0, 0]

        min_x = np.min(xs)
        max_x = np.max(xs)

        limit_x = min_x + 0.45 * (max_x - min_x)

        # Минимальная глубина дефекта
        min_depth = max(10, frame.shape[0] * 0.025)

        defects_num = 0

        for start, end, far, depth in defects.reshape(-1, 4):
            p1 = contour[int(start)][0].astype(np.float32)
            p2 = contour[int(end)][0].astype(np.float32)
            p3 = contour[int(far)][0].astype(np.float32)

            depth = float(depth) / 256.0

            # Векторы от углубления к краям руки
            v1 = p1 - p3
            v2 = p2 - p3

            length1 = np.linalg.norm(v1)
            length2 = np.linalg.norm(v2)

            if length1 == 0 or length2 == 0:
                continue

            # Угол между векторами
            cos_angle = np.dot(v1, v2) / (length1 * length2)
            cos_angle = np.clip(cos_angle, -1, 1)

            angle = np.degrees(np.arccos(cos_angle))

            x, y = int(p3[0]), int(p3[1])

            if depth > min_depth and angle < 90 and x > limit_x:
                # Зелёная точка -- подходящий дефект
                cv2.circle(result, (x, y), 5, (0, 255, 0), -1)

                defects_num += 1

            elif depth > 10:
                # Красная точка -- отброшенный дефект
                cv2.circle(result, (x, y), 5, (0, 0, 255), -1)

        return result, defects_num

    def process_frame(self, frame, lth, hth):
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        mask = cv2.inRange(hsv, lth, hth)

        kernel = np.ones((5, 5), np.uint8)

        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask)

        hand_mask = np.zeros_like(mask)

        if num_labels > 1:
            largest_label = np.argmax(stats[1:, cv2.CC_STAT_AREA]) + 1
            hand_mask[labels == largest_label] = 255

        hand_mask = fill_holes(hand_mask)

        result, defects_num = self.get_defects(frame, hand_mask)

        if defects_num > 0:
            fingers_num = min(5, defects_num + 1)
        else:
            fingers_num = 0

        cv2.imshow("hand mask", hand_mask)
        cv2.imshow("convexity defects", result)

        return int(fingers_num)


class MorphologyCounter(FrameProcessor):
    def __init__(self):
        super().__init__()

    def get_fingers_mask(self, hand_mask):
        h, w = hand_mask.shape

        size = max(15, int(h * 0.07))

        if size % 2 == 0:
            size += 1

        kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE, (size, size)
        )

        top_hat = cv2.morphologyEx(
            hand_mask, cv2.MORPH_TOPHAT, kernel
        )

        black_hat = cv2.morphologyEx(
            hand_mask, cv2.MORPH_BLACKHAT, kernel
        )

        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(top_hat)

        fingers_mask = np.zeros_like(hand_mask)

        min_area = h * w * 0.00015

        for i in range(1, num_labels):
            area = stats[i, cv2.CC_STAT_AREA]

            if area < min_area:
                continue

            fingers_mask[labels == i] = 255

        return top_hat, black_hat, fingers_mask

    def count_fingers(self, hand_mask, fingers_mask):
        num_labels, _, _, _ = cv2.connectedComponentsWithStats(fingers_mask)

        fingers_num = min(5, num_labels - 1)

        return fingers_num

    def process_frame(self, frame, lth, hth):
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        mask = cv2.inRange(hsv, lth, hth)

        kernel = np.ones((5, 5), np.uint8)

        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask)

        hand_mask = np.zeros_like(mask)

        if num_labels > 1:
            largest_label = np.argmax(stats[1:, cv2.CC_STAT_AREA]) + 1
            hand_mask[labels == largest_label] = 255

        hand_mask = fill_holes(hand_mask)

        top_hat, black_hat, fingers_mask = self.get_fingers_mask(hand_mask)

        fingers_num = self.count_fingers(hand_mask, fingers_mask)

        result = frame.copy()

        stages = cv2.hconcat([top_hat, black_hat, fingers_mask])

        stages = cv2.resize(
            stages, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_NEAREST
        )

        cv2.imshow("morphology result", result)
        cv2.imshow("top hat | black hat | fingers", stages)

        return int(fingers_num)

In [85]:
cam = cv2.VideoCapture(video_file)

finger_counter = SkeletonizationCounter()

skeletonization_fingers_num = finger_counter.processing_loop(cam, lth, hth)

print(skeletonization_fingers_num)

cam.release()
cv2.destroyAllWindows()
cv2.waitKey(100)

Cannot read frame
Exiting loop
[5, 5, 1, 0, 1, 5, 5, 5, 1, 0, 0, 5, 5, 5, 5, 4, 4, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 3, 3, 4, 4, 4, 3, 3, 5, 4, 4, 4, 4, 4, 4, 5, 3, 2, 2, 2, 2, 2, 2, 2, 2, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 0, 1, 0, 0, 1, 0]


-1

In [86]:
cam = cv2.VideoCapture(video_file)

finger_counter = ConvexityCounter()

convexity_fingers_num = finger_counter.processing_loop(cam, lth, hth)

print(convexity_fingers_num)

cam.release()
cv2.destroyAllWindows()
cv2.waitKey(100)

Cannot read frame
Exiting loop
[5, 5, 0, 0, 0, 4, 5, 5, 0, 0, 0, 5, 5, 5, 5, 4, 3, 3, 3, 3, 2, 0, 0, 0, 2, 2, 2, 2, 2, 2, 0, 0, 0, 2, 2, 2, 2, 2, 2, 3, 3, 4, 4, 4, 3, 3, 5, 5, 5, 5, 5, 5, 5, 5, 2, 0, 0, 0, 0, 2, 2, 2, 2, 0, 2, 0, 0, 0, 0, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0]


-1

In [94]:
cam = cv2.VideoCapture(video_file)

finger_counter = MorphologyCounter()

morphology_fingers_num = finger_counter.processing_loop(cam, lth, hth)

print(morphology_fingers_num)

cam.release()
cv2.destroyAllWindows()
cv2.waitKey(100)

Cannot read frame
Exiting loop
[5, 5, 1, 0, 0, 5, 5, 5, 2, 0, 0, 5, 5, 5, 5, 4, 3, 3, 3, 3, 3, 1, 1, 1, 3, 3, 3, 3, 3, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 3, 3, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 4, 0, 1, 1, 1, 1, 1]


-1

***

<h2 style="color:#A7BD3F;">Секция 4: оценивание</h2>

Решение, дающее значения метрики *accuracy* *0.5* и более, оцениваются полным баллом с линейной интерполяцией в области меньших значений.

Выполните эту клетку для получения своей оценки, изучите, как она была посчитана. Если в процессе выполнения задания появились вопросы или проблемы, пишите в чат или @elijahmipt в телеграме.

In [98]:
reference_fingers_num = [5, 5, 1, 0, 0, 5, 5, 5, 0, 0, 0, 5, 5, 5, 5, 4, 3, 3,\
                         3, 3, 3, 3, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 2, 2,\
                         2, 2, 2, 2, 3, 4, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5,\
                         2, 2, 2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2,\
                         2, 2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 2, 1, 1, 1, 1, 1, 1,\
                         3, 4, 0, 0, 0, 1]

max_grade = 100

skeletonization_corr_num = 0
for r, s in zip(reference_fingers_num, skeletonization_fingers_num):
    if (r == s):
        skeletonization_corr_num += 1

acc = skeletonization_corr_num / len(reference_fingers_num)
grade = min(acc * 2, 1) * max_grade
print("Your grade is ", "\033[92m{}\033[0m".format(str(int(grade)) +\
        " out of " + str(max_grade) + "; " + str(skeletonization_corr_num) + " frames out of "
        + str(len(reference_fingers_num)) + " in skeletonization"))

convexity_corr_num = 0
for r, s in zip(reference_fingers_num, convexity_fingers_num):
    if (r == s):
        convexity_corr_num += 1
acc = convexity_corr_num / len(reference_fingers_num)
grade = min(acc * 2, 1) * max_grade
print("Your grade is ", "\033[92m{}\033[0m".format(str(int(grade)) +\
        " out of " + str(max_grade) + "; " + str(convexity_corr_num) + " frames out of "
        + str(len(reference_fingers_num)) + " in convexity"))

morphology_corr_num = 0
for r, s in zip(reference_fingers_num, morphology_fingers_num):
    if (r == s):
        morphology_corr_num += 1
acc = morphology_corr_num / len(reference_fingers_num)
grade = min(acc * 2, 1) * max_grade
print("Your grade is ", "\033[92m{}\033[0m".format(str(int(grade)) +\
        " out of " + str(max_grade) + "; " + str(morphology_corr_num) + " frames out of "
        + str(len(reference_fingers_num)) + " in morphology"))

Your grade is  100 out of 100; 48 frames out of 96 in skeletonization
Your grade is  100 out of 100; 51 frames out of 96 in convexity
Your grade is  100 out of 100; 61 frames out of 96 in morphology
